In [1]:
import os
import re
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
folders = [
    "data/documents",
    "models",
    "src"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


In [3]:
document_folder = "data/documents"

document_files = [
    filename
    for filename in os.listdir(document_folder)
    if filename.endswith(".txt")
]

print("Available documents:", len(document_files))

for filename in document_files:
    print("-", filename)

Available documents: 3
- machine_learning.txt
- python_basics.txt
- university_rules.txt


In [4]:
def load_documents(folder_path):
    loaded_documents = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)

            with open(file_path, "r", encoding="utf-8") as file:
                content = file.read()

            if content.strip():
                loaded_documents.append({
                    "filename": filename,
                    "content": content
                })

    return loaded_documents


loaded_documents = load_documents(document_folder)

print("Total documents loaded:", len(loaded_documents))

for document in loaded_documents:
    print("\nFilename:", document["filename"])
    print("Characters:", len(document["content"]))

Total documents loaded: 3

Filename: machine_learning.txt
Characters: 599

Filename: python_basics.txt
Characters: 531

Filename: university_rules.txt
Characters: 457


In [5]:
def chunk_documents(documents, chunk_size=3):
    chunks = []

    for document in documents:
        sentences = re.split(
            r'(?<=[.!?])\s+',
            document["content"].strip()
        )

        sentences = [
            sentence.strip()
            for sentence in sentences
            if sentence.strip()
        ]

        for i in range(0, len(sentences), chunk_size):
            chunk_text = " ".join(
                sentences[i:i + chunk_size]
            )

            chunks.append({
                "source": document["filename"],
                "chunk_id": len(chunks),
                "text": chunk_text
            })

    return chunks


chunks = chunk_documents(loaded_documents, chunk_size=3)

print("Total chunks created:", len(chunks))

for chunk in chunks:
    print(f"\nChunk ID: {chunk['chunk_id']}")
    print("Source:", chunk["source"])
    print("Text:", chunk["text"])

Total chunks created: 8

Chunk ID: 0
Source: machine_learning.txt
Text: Introduction to Machine Learning

Machine Learning is a branch of artificial intelligence
that enables computers to learn patterns from data. Supervised learning uses labeled training data. Its common tasks include classification and regression.

Chunk ID: 1
Source: machine_learning.txt
Text: Unsupervised learning works with unlabeled data. Clustering is one of its common techniques. A dataset is commonly divided into training and testing sets.

Chunk ID: 2
Source: machine_learning.txt
Text: The training set is used to teach the model, while the
testing set evaluates its performance. Common classification evaluation metrics include accuracy,
precision, recall, F1-score, and ROC-AUC.

Chunk ID: 3
Source: python_basics.txt
Text: Python Programming Basics

Python is a high-level, interpreted programming language. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradi

In [6]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Embedding model loaded successfully.


In [7]:
chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings shape:", chunk_embeddings.shape)
print("Total embedded chunks:", len(chunk_embeddings))

Embeddings shape: (8, 384)
Total embedded chunks: 8


In [8]:
def retrieve_passages(question, top_k=3):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in top_indices:
        result = chunks[index].copy()
        result["score"] = float(scores[index])
        results.append(result)

    return results


print("Semantic retrieval function created successfully.")

Semantic retrieval function created successfully.


In [9]:
question = "Who created Python and when was it released?"

results = retrieve_passages(question)

print("Question:", question)

for result in results:
    print("\nSource:", result["source"])
    print("Chunk ID:", result["chunk_id"])
    print("Similarity:", round(result["score"], 4))
    print("Passage:", result["text"])
    print("-" * 60)

Question: Who created Python and when was it released?

Source: python_basics.txt
Chunk ID: 3
Similarity: 0.6874
Passage: Python Programming Basics

Python is a high-level, interpreted programming language. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including
object-oriented, procedural, and functional programming.
------------------------------------------------------------

Source: python_basics.txt
Chunk ID: 4
Similarity: 0.3999
Passage: Variables are used to store data values. Python has built-in data types such as integers, floats,
strings, lists, tuples, dictionaries, and sets. Functions are reusable blocks of code defined using the
def keyword.
------------------------------------------------------------

Source: python_basics.txt
Chunk ID: 5
Similarity: 0.1549
Passage: They help organize programs and reduce repetition.
------------------------------------------------------------


In [10]:
question = "What is the capital of France?"

results = retrieve_passages(question)

print("Question:", question)

for result in results:
    print("\nSource:", result["source"])
    print("Similarity:", round(result["score"], 4))
    print("Passage:", result["text"])
    print("-" * 60)

Question: What is the capital of France?

Source: machine_learning.txt
Similarity: 0.0902
Passage: The training set is used to teach the model, while the
testing set evaluates its performance. Common classification evaluation metrics include accuracy,
precision, recall, F1-score, and ROC-AUC.
------------------------------------------------------------

Source: python_basics.txt
Similarity: 0.0586
Passage: Python Programming Basics

Python is a high-level, interpreted programming language. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including
object-oriented, procedural, and functional programming.
------------------------------------------------------------

Source: machine_learning.txt
Similarity: 0.0554
Passage: Introduction to Machine Learning

Machine Learning is a branch of artificial intelligence
that enables computers to learn patterns from data. Supervised learning uses labeled training data. Its common tasks i

In [11]:
SIMILARITY_THRESHOLD = 0.40

print("Similarity threshold:", SIMILARITY_THRESHOLD)

Similarity threshold: 0.4


In [12]:
def extract_best_sentence(question, passage):
    sentences = re.split(
        r'(?<=[.!?])\s+',
        passage.strip()
    )

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    sentence_embeddings = embedding_model.encode(
        sentences,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        question_embedding,
        sentence_embeddings
    )[0]

    best_index = np.argmax(scores)

    return sentences[best_index]

In [13]:
question = "Who created Python and when was it released?"

passage = results[1]["text"]

answer = extract_best_sentence(question, passage)

print("Question:", question)
print("Extracted sentence:", answer)

Question: Who created Python and when was it released?
Extracted sentence: Python Programming Basics

Python is a high-level, interpreted programming language.


In [14]:
python_results = retrieve_passages(
    "Who created Python and when was it released?"
)

for result in python_results:
    if result["source"] == "python_basics.txt":
        print(result["text"])
        break

Python Programming Basics

Python is a high-level, interpreted programming language. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including
object-oriented, procedural, and functional programming.


In [15]:
passage = python_results[0]["text"]

sentences = re.split(
    r'(?<=[.!?])\s+',
    passage.strip()
)

for i, sentence in enumerate(sentences):
    print(f"{i}: {sentence.strip()}")

0: Python Programming Basics

Python is a high-level, interpreted programming language.
1: It was created by Guido van Rossum and first released in 1991.
2: Python supports multiple programming paradigms, including
object-oriented, procedural, and functional programming.


In [16]:
for sentence in sentences:
    if "created by" in sentence.lower():
        print("Matching sentence:", sentence.strip())

Matching sentence: It was created by Guido van Rossum and first released in 1991.


In [17]:
patterns = [
    "created by",
    "first released",
    "is used to",
    "is a branch of",
    "requires",
    "must"
]

for sentence in sentences:
    for pattern in patterns:
        if pattern in sentence.lower():
            print("Pattern:", pattern)
            print("Sentence:", sentence.strip())
            print("-" * 40)

Pattern: created by
Sentence: It was created by Guido van Rossum and first released in 1991.
----------------------------------------
Pattern: first released
Sentence: It was created by Guido van Rossum and first released in 1991.
----------------------------------------


In [18]:
try:
    import transformers
    print("Transformers version:", transformers.__version__)
except ImportError:
    print("Transformers is not installed.")

Transformers version: 4.57.1


In [19]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="deepset/minilm-uncased-squad2",
    tokenizer="deepset/minilm-uncased-squad2"
)

print("Question-answering model loaded successfully.")

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Question-answering model loaded successfully.


In [20]:
%pip install transformers==4.57.1

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [21]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 4.57.1


In [22]:
question = "Who created Python and when was it released?"

context = python_results[0]["text"]

qa_result = qa_pipeline(
    question=question,
    context=context
)

print("Question:", question)
print("Answer:", qa_result["answer"])
print("Confidence score:", round(qa_result["score"], 4))

Question: Who created Python and when was it released?
Answer: Guido van Rossum
Confidence score: 0.5034


In [23]:
question = "When was Python first released?"

qa_result = qa_pipeline(
    question=question,
    context=python_results[0]["text"]
)

print("Question:", question)
print("Answer:", qa_result["answer"])
print("Confidence score:", round(qa_result["score"], 4))

Question: When was Python first released?
Answer: 1991
Confidence score: 0.9871


In [24]:
question = "What is the capital of France?"

qa_result = qa_pipeline(
    question=question,
    context=python_results[0]["text"]
)

print("Question:", question)
print("Answer:", qa_result["answer"])
print("Confidence score:", round(qa_result["score"], 4))

Question: What is the capital of France?
Answer: Python Programming Basics

Python is a high-level, interpreted programming language
Confidence score: 0.0


In [25]:
CONFIDENCE_THRESHOLD = 0.40

if qa_result["score"] < CONFIDENCE_THRESHOLD:
    answer = "Information not found in provided document"
else:
    answer = qa_result["answer"]

print("Final answer:", answer)

Final answer: Information not found in provided document


In [26]:
question = "Who created Python?"

context = python_results[0]["text"]

qa_result = qa_pipeline(
    question=question,
    context=context
)

if qa_result["score"] < CONFIDENCE_THRESHOLD:
    answer = "Information not found in provided document"
else:
    answer = qa_result["answer"]

print("Question:", question)
print("Answer:", answer)
print("Confidence:", round(qa_result["score"], 4))

Question: Who created Python?
Answer: Guido van Rossum
Confidence: 0.9943


In [27]:
def answer_question(question):
    results = retrieve_passages(question, top_k=3)

    best_result = results[0]

    if best_result["score"] < SIMILARITY_THRESHOLD:
        return {
            "answer": "Information not found in provided document",
            "source": None,
            "chunk_id": None,
            "passage": None
        }

    qa_result = qa_pipeline(
        question=question,
        context=best_result["text"]
    )

    if qa_result["score"] < CONFIDENCE_THRESHOLD:
        answer = "Information not found in provided document"
    else:
        answer = qa_result["answer"]

    return {
        "answer": answer,
        "source": best_result["source"],
        "chunk_id": best_result["chunk_id"],
        "passage": best_result["text"],
        "similarity": best_result["score"],
        "confidence": qa_result["score"]
    }

In [28]:
question = "Who created Python?"

response = answer_question(question)

print("Question:", question)
print("Answer:", response["answer"])
print("Source:", response["source"])
print("Chunk ID:", response["chunk_id"])
print("Similarity:", round(response["similarity"], 4))
print("Confidence:", round(response["confidence"], 4))
print("\nRetrieved passage:")
print(response["passage"])

Question: Who created Python?
Answer: Guido van Rossum
Source: python_basics.txt
Chunk ID: 3
Similarity: 0.6776
Confidence: 0.9943

Retrieved passage:
Python Programming Basics

Python is a high-level, interpreted programming language. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including
object-oriented, procedural, and functional programming.


In [29]:
question = "What is the capital of France?"

response = answer_question(question)

print("Question:", question)
print("Answer:", response["answer"])
print("Source:", response["source"])

Question: What is the capital of France?
Answer: Information not found in provided document
Source: None


In [30]:
question = "What is supervised learning?"

response = answer_question(question)

print("Question:", question)
print("Answer:", response["answer"])
print("Source:", response["source"])
print("Chunk ID:", response["chunk_id"])
print("Similarity:", round(response["similarity"], 4))
print("Confidence:", round(response["confidence"], 4))

Question: What is supervised learning?
Answer: uses labeled training data
Source: machine_learning.txt
Chunk ID: 0
Similarity: 0.6732
Confidence: 0.4311


In [31]:
question = "What is the minimum attendance requirement?"

response = answer_question(question)

print("Question:", question)
print("Answer:", response["answer"])
print("Source:", response["source"])
print("Chunk ID:", response["chunk_id"])
print("Similarity:", round(response["similarity"], 4))
print("Confidence:", round(response["confidence"], 4))

Question: What is the minimum attendance requirement?
Answer: 75 percent
Source: university_rules.txt
Chunk ID: 6
Similarity: 0.686
Confidence: 0.9345


In [32]:
question = "Which programming language is faster, Python or Java?"

response = answer_question(question)

print("Question:", question)
print("Answer:", response["answer"])
print("Source:", response["source"])

Question: Which programming language is faster, Python or Java?
Answer: Information not found in provided document
Source: python_basics.txt


In [33]:
def answer_question(question):
    results = retrieve_passages(question, top_k=3)
    best_result = results[0]

    if best_result["score"] < SIMILARITY_THRESHOLD:
        return {
            "answer": "Information not found in provided document",
            "source": None,
            "chunk_id": None,
            "passage": None
        }

    qa_result = qa_pipeline(
        question=question,
        context=best_result["text"]
    )

    if qa_result["score"] < CONFIDENCE_THRESHOLD:
        return {
            "answer": "Information not found in provided document",
            "source": None,
            "chunk_id": None,
            "passage": None
        }

    return {
        "answer": qa_result["answer"],
        "source": best_result["source"],
        "chunk_id": best_result["chunk_id"],
        "passage": best_result["text"],
        "similarity": best_result["score"],
        "confidence": qa_result["score"]
    }

In [34]:
import ipywidgets as widgets

print("ipywidgets is ready.")

ipywidgets is ready.


In [35]:
question_input = widgets.Textarea(
    placeholder="Ask a question about your documents...",
    description="Question:",
    layout=widgets.Layout(width="80%", height="80px")
)

ask_button = widgets.Button(
    description="Ask Question",
    button_style="primary",
    icon="search"
)

output_area = widgets.Output()

print("Interface components created successfully.")

Interface components created successfully.


In [36]:
def on_ask_button_clicked(button):
    question = question_input.value.strip()

    with output_area:
        output_area.clear_output()

        if not question:
            print("Please enter a question.")
            return

        response = answer_question(question)

        print("ANSWER")
        print(response["answer"])

        if response["source"] is not None:
            print("\nSOURCE")
            print(response["source"])
            print("Chunk ID:", response["chunk_id"])

            print("\nEVIDENCE PASSAGE")
            print(response["passage"])


ask_button.on_click(on_ask_button_clicked)

print("Button event handler connected successfully.")

Button event handler connected successfully.


In [38]:
display(
    widgets.VBox([
        widgets.HTML("<h2>Document Question-Answering Assistant</h2>"),
        widgets.HTML("<p>Ask questions based on your uploaded documents.</p>"),
        question_input,
        ask_button,
        output_area
    ])
)